# Assignment 8: YOLOv3-Tiny Real-Time Object Detection Pipeline
**Author:** Kristian Hudák  
**Course:** AI Olympiad (OAI) Preparation Course

This notebook demonstrates deploying a fine-tuned lightweight YOLOv3-Tiny object detector for video inference:
1. Setup and environment configuration for PyTorch CUDA inference
2. Loading custom pre-trained YOLOv3-Tiny weights (`OVLADACE_detector_yolov3_tiny.pth` - remote control detector)
3. Image preprocessing pipeline: letterboxing, color-space conversion, blob normalization
4. Post-processing: Non-Maximum Suppression (NMS), bounding box coordinate rescaling, annotation rendering
5. Frame-by-frame video stream inference with OpenCV VideoWriter


In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchsummary import summary

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [3]:
# upload your model
from google.colab import files
uploaded = files.upload('OVLADACE_detector_yolov3_tiny.pth')
modelfile = list(uploaded.keys())[0]
print(modelfile)

Saving OVLADACE_detector_yolov3_tiny.pth to OVLADACE_detector_yolov3_tiny.pth/OVLADACE_detector_yolov3_tiny.pth
OVLADACE_detector_yolov3_tiny.pth/OVLADACE_detector_yolov3_tiny.pth


In [18]:
!git clone https://github.com/Lornatang/YOLOv3-PyTorch.git

Cloning into 'YOLOv3-PyTorch'...
remote: Enumerating objects: 3667, done.
remote: Counting objects: 100% (1782/1782), done.
remote: Compressing objects: 100% (597/597), done.
remote: Total 3667 (delta 1230), reused 1575 (delta 1154), pack-reused 1885 (from 1)
Receiving objects: 100% (3667/3667), 1.39 MiB | 24.07 MiB/s, done.
Resolving deltas: 100% (2454/2454), done.


In [19]:
!ls YOLOv3-PyTorch

configs  figure   model_configs  requirements.txt  scripts   tools
data	 LICENSE  README.md	 results	   setup.py  yolov3_pytorch


In [20]:
!ln -sf YOLOv3-PyTorch/yolov3_pytorch yolov3_pytorch
!ln -sf YOLOv3-PyTorch/tools tools
!ln -sf YOLOv3-PyTorch/configs configs
!ln -sf YOLOv3-PyTorch/model_configs model_configs

In [21]:
!mv tools/export.py tools/export.py.org
!mv yolov3_pytorch/engine/trainer.py yolov3_pytorch/engine/trainer.py.org
!wget http://www.agentspace.org/download/yolov3_pytorch_patch.zip
!unzip -o yolov3_pytorch_patch.zip
!rm yolov3_pytorch_patch.zip

--2025-12-04 19:30:56--  http://www.agentspace.org/download/yolov3_pytorch_patch.zip
Resolving www.agentspace.org (www.agentspace.org)... 62.168.101.9
Connecting to www.agentspace.org (www.agentspace.org)|62.168.101.9|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.agentspace.org/download/yolov3_pytorch_patch.zip [following]
--2025-12-04 19:30:57--  https://www.agentspace.org/download/yolov3_pytorch_patch.zip
Connecting to www.agentspace.org (www.agentspace.org)|62.168.101.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7567 (7.4K) [application/zip]
Saving to: ‘yolov3_pytorch_patch.zip’

yolov3_pytorch_patc 100%[===================>]   7.39K  --.-KB/s    in 0s      

2025-12-04 19:30:58 (2.34 GB/s) - ‘yolov3_pytorch_patch.zip’ saved [7567/7567]

Archive:  yolov3_pytorch_patch.zip
  inflating: tools/export.py         
  inflating: tools/export.py.org     
  inflating: yolov3_pytorch/engine/trainer.py  


In [28]:
# load the model
!pip install thop
import yolov3_pytorch
from yolov3_pytorch.utils import scale_coords, xyxy2xywh, non_max_suppression, plot_one_box
from yolov3_pytorch.data.data_augment import letterbox
model_path = modelfile # Use the correct path from the uploaded file
model = torch.load(model_path, weights_only=False).to(device)
model.eval()

Darknet(
  (module_list): ModuleList(
    (0): Sequential(
      (Conv2d): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (BatchNorm2d): BatchNorm2d(16, eps=0.0001, momentum=0.03, affine=True, track_running_stats=True)
      (activation): LeakyReLU(negative_slope=0.1, inplace=True)
    )
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Sequential(
      (Conv2d): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (BatchNorm2d): BatchNorm2d(32, eps=0.0001, momentum=0.03, affine=True, track_running_stats=True)
      (activation): LeakyReLU(negative_slope=0.1, inplace=True)
    )
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Sequential(
      (Conv2d): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (BatchNorm2d): BatchNorm2d(64, eps=0.0001, momentum=0.03, affine=True, track_running_stats=True)
 

In [34]:
names = ['OVLADACE']

In [35]:
# define the image processing
def process_image(frame, conf_thresh=0.1, iou_thresh=0.45):
    img_size = 416
    img, _, _ = letterbox(frame,new_shape=img_size)
    img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    blob = cv2.dnn.blobFromImage(img,1.0/255)
    blob = torch.tensor(blob)
    blob = blob.to(device)
    with torch.no_grad():
        output, _ = model(blob, False)
    detections = non_max_suppression(output, conf_thresh, iou_thresh)[0]
    if detections is None:
        return frame
    detections[:, :4] = scale_coords(blob.shape[2:], detections[:, :4], frame.shape).round()
    disp = np.copy(frame)
    for detection in detections:
        *xyxy, confidence, classid  = detection
        plot_one_box(xyxy, disp, label=names[classid.int().item()], color=(0,0,255))
    return disp


In [39]:
# upload your video
uploaded = files.upload('video_619604.mp4')
videofile = list(uploaded.keys())[0]
print(videofile)

Saving IMG_6219.mp4 to video_619604.mp4/IMG_6219.mp4
video_619604.mp4/IMG_6219.mp4


In [40]:
# process video
resultfile = 'result.avi'
video = cv2.VideoCapture(videofile)
fps = video.get(cv2.CAP_PROP_FPS)
hasFrame, frame = video.read()
out = cv2.VideoWriter()
out.open(resultfile,cv2.VideoWriter_fourcc('M','J','P','G'),fps,(frame.shape[1],frame.shape[0]))
while True:
    result = process_image(frame)
    out.write(result)
    hasFrame, frame = video.read()
    if not hasFrame:
        break
out.release()

In [41]:
# download result
files.download(resultfile)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>